# 섹터 데이터 수집 (morning-report ADR-003 Amendment)

**목적**: ADR-003 섹터 강도 산식 구현에 필요한 데이터를 수집해서
Google Drive 에 parquet 으로 저장한다.

**주의**: 2026-04-23 Phase B 실행 중 pykrx 인덱스 API 전면 다운 확인.
Amendment 반영 — 데이터 소스 pivot + Weinstein Stage 25점 보류.
상세: `docs/decisions/003-sector-strength-methodology.md` Amendment 2026-04-23.

**수집 내역**:
1. `sector_map.parquet` — KRX KIND 기반 162종목 KSIC + KRX 22업종 매핑 + FDR 시총
2. `stocks_daily.parquet` — 162종목 일봉 OHLCV (최근 2년, pykrx 종목 API 정상)

업종지수 주봉은 pykrx 복구 후 재개 (Section 2 제거).

**사용법**: 런타임 → 모두 실행 (Ctrl+F9). 5-10분 소요.

**산출물 위치**: `MyDrive/morning-report/sector_data/`

---

## 섹션 0: 환경 준비

In [ ]:
# pykrx (종목 OHLCV 용) + pyarrow (parquet) + finance-datareader (시총)
!pip install -q pykrx pyarrow finance-datareader

In [ ]:
import io
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import requests
import yaml
from pykrx import stock
import FinanceDataReader as fdr

print(f'pandas {pd.__version__}')
print(f'fdr {fdr.__version__}')
print(f'python {sys.version.split()[0]}')

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = Path('/content/drive/MyDrive/morning-report/sector_data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'산출물 저장 경로: {OUTPUT_DIR}')

In [ ]:
# 기간: 최근 2년 (종목 일봉용)
END_DATE = datetime.now().strftime('%Y%m%d')
START_DATE = (datetime.now() - timedelta(days=365 * 2 + 10)).strftime('%Y%m%d')
print(f'수집 기간: {START_DATE} ~ {END_DATE}')

In [ ]:
# UZymn 브랜치 clone (이미 있으면 최신으로 리셋)
import subprocess

REPO_DIR = Path('/content/morning-report')
BRANCH = 'claude/session-start-UZymn'
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '-b', BRANCH, '--depth', '1',
        'https://github.com/gengar200005/morning-report.git',
        str(REPO_DIR),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'FETCH_HEAD'], check=True)

# UNIVERSE 로딩
sys.path.insert(0, str(REPO_DIR / 'backtest'))
if 'universe' in sys.modules:
    import importlib, universe
    importlib.reload(universe)
    from universe import UNIVERSE
else:
    from universe import UNIVERSE

UNIVERSE_TICKERS = [t for t, _ in UNIVERSE]
UNIVERSE_NAMES = dict(UNIVERSE)
universe_set = set(UNIVERSE_TICKERS)
print(f'UNIVERSE: {len(UNIVERSE)}종목')

---

## 섹션 1: 섹터 매핑 (KRX KIND + FDR + KSIC→22업종)

1. **KRX KIND 상장법인목록** 파싱 → 회사/종목코드/업종(KSIC)/시장구분
2. **FinanceDataReader** `StockListing('KRX')` → 시가총액(Marcap)
3. `reports/sector_overrides.yaml` 로드 → KSIC → KOSPI 22업종 매핑 적용
4. 최종 산출: `sector_map.parquet`
   - 컬럼: `ticker, name, market, ksic, sector, marcap`
   - 매칭 실패 종목은 ksic/sector=NaN (상폐 등)

In [ ]:
# 1-1. KRX KIND 상장법인목록 다운로드 + 파싱
r = requests.get(
    'http://kind.krx.co.kr/corpgeneral/corpList.do',
    params={'method': 'download', 'searchType': '13'},
    timeout=15,
)
r.raise_for_status()
kind = pd.read_html(io.StringIO(r.text), encoding='euc-kr')[0]
kind['종목코드'] = kind['종목코드'].astype(str).str.zfill(6)
print(f'[KIND] 전체 {len(kind)}사, 컬럼: {list(kind.columns)}')

kind_u = kind[kind['종목코드'].isin(universe_set)][['종목코드', '회사명', '시장구분', '업종']].copy()
kind_u = kind_u.rename(columns={'종목코드': 'ticker', '회사명': 'kind_name', '시장구분': 'market', '업종': 'ksic'})
print(f'[KIND] UNIVERSE 매칭: {len(kind_u)}/{len(UNIVERSE)}')
print(f'[KIND] 고유 KSIC: {kind_u["ksic"].nunique()}종')

In [ ]:
# 1-2. FDR 시가총액 수집
fdr_krx = fdr.StockListing('KRX')[['Code', 'Marcap']].copy()
fdr_krx['Code'] = fdr_krx['Code'].astype(str).str.zfill(6)
fdr_krx = fdr_krx.rename(columns={'Code': 'ticker', 'Marcap': 'marcap'})

fdr_u = fdr_krx[fdr_krx['ticker'].isin(universe_set)]
print(f'[FDR] UNIVERSE 매칭: {len(fdr_u)}/{len(UNIVERSE)}')
print(f'[FDR] 시총 합계: {fdr_u["marcap"].sum() / 1e12:.1f}조')

In [ ]:
# 1-3. KSIC → KOSPI 22업종 매핑 로드 + 적용
overrides_path = REPO_DIR / 'reports' / 'sector_overrides.yaml'
with open(overrides_path, encoding='utf-8') as f:
    overrides = yaml.safe_load(f)

ksic_map = overrides.get('ksic_to_kospi22', {})
ticker_map = overrides.get('ticker_overrides', {}) or {}
print(f'[매핑] KSIC 규칙 {len(ksic_map)}개, 종목 오버라이드 {len(ticker_map)}개')

def classify(row):
    # 1순위: 종목 단위 오버라이드
    if row['ticker'] in ticker_map:
        return ticker_map[row['ticker']]
    # 2순위: KSIC 자동 매핑
    return ksic_map.get(row['ksic'])

kind_u['sector'] = kind_u.apply(classify, axis=1)

unmapped_ksic = kind_u[kind_u['sector'].isna()]['ksic'].value_counts()
if len(unmapped_ksic):
    print(f'\n[!] KSIC 매핑 누락 {len(unmapped_ksic)}종:')
    print(unmapped_ksic)
    print('→ reports/sector_overrides.yaml 의 ksic_to_kospi22 에 추가 필요')
else:
    print('\n[OK] 모든 KSIC 매핑됨')

In [ ]:
# 1-4. 최종 sector_map 빌드 + 저장
# UNIVERSE 기준 left join → 상폐 등으로 KIND/FDR 없는 종목도 포함 (ksic/sector/marcap=NaN)
base = pd.DataFrame(UNIVERSE, columns=['ticker', 'name'])
sector_map = (
    base
    .merge(kind_u[['ticker', 'market', 'ksic', 'sector']], on='ticker', how='left')
    .merge(fdr_u[['ticker', 'marcap']], on='ticker', how='left')
)

print(f'sector_map: {len(sector_map)}행')
print(f'  KIND 매칭:  {sector_map["ksic"].notna().sum()}')
print(f'  섹터 매핑:  {sector_map["sector"].notna().sum()}')
print(f'  시총 매칭:  {sector_map["marcap"].notna().sum()}')

missing = sector_map[sector_map['sector'].isna()][['ticker', 'name', 'ksic']]
if len(missing):
    print(f'\n--- 섹터 미매핑 {len(missing)}종목 ---')
    print(missing.to_string(index=False))

print('\n--- KRX 22업종 분포 ---')
print(sector_map['sector'].value_counts(dropna=False))

out_path = OUTPUT_DIR / 'sector_map.parquet'
sector_map.to_parquet(out_path, index=False)
print(f'\n저장: {out_path} ({out_path.stat().st_size / 1024:.1f} KB)')

---

## 섹션 2: 162종목 일봉 OHLCV

UNIVERSE 162종목의 **일봉** OHLCV (최근 2년). pykrx 종목 API 는 정상 작동.
Breadth(% above MA50) 및 IBD 6M 계산 모두 일봉 기준.

산출: `stocks_daily.parquet`
- 컬럼: `날짜, ticker, name, 시가, 고가, 저가, 종가, 거래량, 거래대금, 등락률`
- 상폐 종목은 자동 누락 (실패 리스트 표시)

※ Section 2 (업종지수 주봉) 는 pykrx 인덱스 API 복구 시 재도입.

In [ ]:
# 2-1. 162종목 일봉 OHLCV 수집 (rate limit: 0.2초/호출 = 약 35초)
stock_frames = []
stock_fails = []
for i, (ticker, nm) in enumerate(UNIVERSE):
    try:
        df = stock.get_market_ohlcv_by_date(START_DATE, END_DATE, ticker)
    except Exception as e:
        stock_fails.append((ticker, nm, str(e)))
        continue
    if df is None or df.empty:
        stock_fails.append((ticker, nm, 'empty'))
        continue
    df = df.reset_index()
    df['ticker'] = ticker
    df['name'] = nm
    stock_frames.append(df)
    if (i + 1) % 20 == 0:
        print(f'  진행: {i + 1}/{len(UNIVERSE)}')
    time.sleep(0.2)

stocks_daily = pd.concat(stock_frames, ignore_index=True)
stocks_daily['날짜'] = pd.to_datetime(stocks_daily['날짜'])
print(f'\n종목 일봉 수집: {len(stocks_daily):,}행, {stocks_daily["ticker"].nunique()}종목')
print(f'기간: {stocks_daily["날짜"].min().date()} ~ {stocks_daily["날짜"].max().date()}')
if stock_fails:
    print(f'\n실패 {len(stock_fails)}종목:')
    for t, n, e in stock_fails[:10]:
        print(f'  {t} {n}: {e}')
    if len(stock_fails) > 10:
        print(f'  ... 외 {len(stock_fails) - 10}개')

In [ ]:
# 2-2. parquet 저장
out_path = OUTPUT_DIR / 'stocks_daily.parquet'
stocks_daily.to_parquet(out_path, index=False)
print(f'저장: {out_path}')
print(f'파일 크기: {out_path.stat().st_size / 1024 / 1024:.2f} MB')

---

## 섹션 3: 완료 확인

산출물 2개(sector_map, stocks_daily) 확인. 이 셀까지 정상이면 Phase B 완료.

In [ ]:
# 3. 산출물 검증 (2개)
expected = ['sector_map.parquet', 'stocks_daily.parquet']
print(f'{"파일":<28} {"크기":>10}  {"행":>10}  {"고유키":>12}')
print('-' * 68)
for fname in expected:
    p = OUTPUT_DIR / fname
    if not p.exists():
        print(f'{fname:<28} [MISSING]')
        continue
    size_kb = p.stat().st_size / 1024
    df = pd.read_parquet(p)
    if fname == 'sector_map.parquet':
        key = f'{df["ticker"].nunique()}종목'
    else:
        key = f'{df["ticker"].nunique()}종목'
    print(f'{fname:<28} {size_kb:>8.1f} KB  {len(df):>10,}  {key:>12}')

print('\n[OK] Phase B 완료 — Drive 에서 parquet 2개 확인하고 Claude 에 알려주세요.')